In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
token = os.getenv("KAGGLE_API_TOKEN")

# without dotenv
#os.environ["KAGGLE_API_TOKEN"] = "KGAT_..."

In [ ]:
!kaggle competitions download -c estonian-football-object-detection
!unzip -q estonian-football-object-detection.zip

In [ ]:
import xml.etree.ElementTree as ET
from pathlib import Path
import shutil
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
import csv


PROJECT_ROOT  = Path(".")
TRAIN_IMG_DIR = PROJECT_ROOT / "train" / "images"
TRAIN_ANN_DIR = PROJECT_ROOT / "train" / "annotations"
YOLO_DIR      = PROJECT_ROOT / "yolo_dataset"

CLASS_MAP = {"Player": 0, "Referee": 1, "Goalkeeper": 2}
IMG_W, IMG_H  = 1920, 1080

# Create output dirs
for split in ["train", "val"]:
    (YOLO_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

def voc_to_yolo(xmin, ymin, xmax, ymax):
    cx = ((xmin + xmax) / 2) / IMG_W
    cy = ((ymin + ymax) / 2) / IMG_H
    w  = (xmax - xmin) / IMG_W
    h  = (ymax - ymin) / IMG_H
    return cx, cy, w, h

def parse_and_convert(xml_path):
    labels = []
    for obj in ET.parse(xml_path).iter("object"):
        cls = obj.find("name").text
        if cls not in CLASS_MAP:
            continue
        bb   = obj.find("bndbox")
        xmin = int(bb.find("xmin").text)
        ymin = int(bb.find("ymin").text)
        xmax = int(bb.find("xmax").text)
        ymax = int(bb.find("ymax").text)
        cx, cy, w, h = voc_to_yolo(xmin, ymin, xmax, ymax)
        labels.append(f"{CLASS_MAP[cls]} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
    return labels

def process_split(xml_list, split):
    for xml_path in xml_list:
        stem = xml_path.stem
        # Copy image
        for ext in [".jpg", ".jpeg", ".png"]:
            src_img = TRAIN_IMG_DIR / (stem + ext)
            if src_img.exists():
                shutil.copy(src_img, YOLO_DIR / "images" / split / (stem + ext))
                break
        # Write label
        labels = parse_and_convert(xml_path)
        with open(YOLO_DIR / "labels" / split / f"{stem}.txt", "w") as f:
            f.write("\n".join(labels))

# 90/10 train/val split
all_xmls = sorted(TRAIN_ANN_DIR.glob("*.xml"))
train_xmls, val_xmls = train_test_split(all_xmls, test_size=0.1, random_state=42)

process_split(train_xmls, "train")
process_split(val_xmls,   "val")
print(f"Done: {len(train_xmls)} train / {len(val_xmls)} val samples")

In [ ]:

yaml_content = f"""
path: {Path("yolo_dataset").resolve()}
train: images/train
val: images/val

nc: 3
names: ["Player", "Referee", "Goalkeeper"]
"""

with open("football.yaml", "w") as f:
    f.write(yaml_content.strip())

print("football.yaml written.")

In [ ]:

model = YOLO("yolov8n.pt")  

model.train(
    data="football.yaml",
    epochs=50,
    imgsz=1280,
    batch=8,        
    lr0=0.01,
    cos_lr=True,
    mosaic=1.0,
    patience=20,
    project="runs",
    name="football_yolo",
    device=0,       # 0 = first GPU; set to "cpu" if no GPU
)

In [ ]:

model = YOLO("runs/detect/runs/football_yolo/weights/best.pt")

TEST_DIR    = Path("test")
CLASS_NAMES = {0: "Player", 1: "Referee", 2: "Goalkeeper"}

rows = []
for img_path in sorted(TEST_DIR.glob("*.jpg")):
    result = model.predict(
        source=str(img_path),
        imgsz=1280,
        conf=0.25,
        iou=0.45,
        augment=True,   # TTA — free mAP boost
        verbose=False,
    )[0]

    preds = []
    if result.boxes is not None:
        for box in result.boxes:
            cls_name = CLASS_NAMES[int(box.cls.item())]
            x1, y1, x2, y2 = [int(v) for v in box.xyxy[0].tolist()]
            preds.append(f"{cls_name} {x1} {y1} {x2} {y2}")

    rows.append((img_path.name, " ".join(preds) if preds else "Player 960 540 1000 600"))

with open("submission.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["image_name", "predictions"])
    writer.writerows(rows)

print(f"submission.csv written — {len(rows)} images")